# 3.2 — データの選択・抽出とブール論理

分析の問いを「表示する列」と「残す行の条件」へ分け、条件を検証可能なブールマスクとして表します。

## 導入

このNotebookでは、Moodle本文の概念を実際のデータとコードで確かめます。

## このレッスンの到達目標

- 分析の問いを表示列と行条件へ分解できる。
- 比較式からブールマスクを作り、複数条件を正しく組み合わせられる。
- 所属・範囲・欠損を明示した条件を作れる。
- 抽出前後の件数と並び順を確認し、同じ結果を再現できる。

> **学習経路:** 必須：3.2.1〜3.2.6　／　統合練習：3.2.7


## 3.2.1 問いを表示列と行条件へ分ける

「2026年2月と3月について、登録者30人以上で出席率80%未満のセンター名を確認する」なら、表示列、月の所属条件、登録者数の下限、出席率の上限へ分解します。抽出前に必要列が存在するか確認します。

In [ ]:
from pathlib import Path
import pandas as pd


def find_course_data(filename):
    """Find course data without depending on the Notebook start directory."""
    roots = [Path.cwd(), *Path.cwd().parents, Path.home() / "work", Path("/opt/python-lab/course-materials")]
    checked = []
    for root in roots:
        for candidate in (root / "data" / filename, root / filename):
            candidate = candidate.expanduser()
            if candidate in checked:
                continue
            checked.append(candidate)
            if candidate.is_file():
                return candidate
    locations = "\n".join(f"- {path}" for path in checked)
    raise FileNotFoundError(f"Course data file {filename!r} was not found. Checked:\n{locations}")


data_file = find_course_data("learning-centres-practice.csv")
print("Loading:", data_file.resolve())
df = pd.read_csv(data_file, encoding="utf-8", dtype={"centre_id": "string", "month": "string"})
print("Shape:", df.shape)


### 列名で必要な項目を選ぶ

一列はSeries、複数列は列名リストを使うDataFrameです。存在しない列名は`KeyError`になるため、必要列と`df.columns`の差集合を先に確認できます。

In [ ]:
required = {"month", "centre_name", "registered", "attended", "completed"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"必要な列がありません: {sorted(missing)}")

focused = df[["month", "centre_name", "registered", "attended", "completed"]]
print(focused.head(3))


## 3.2.2 locとilocで行・列を選ぶ

`loc[行条件, 列名]`はindexラベルと列名を使い、分析の意味をコードへ残せます。`iloc[行位置, 列位置]`は0から始まる位置を使い、先頭数行の確認などに向きます。indexラベルが0, 1, 2とは限らないため、両者を混同しません。

In [ ]:
print(df.loc[df.index[:2], ["centre_id", "centre_name"]])
print(df.iloc[:2, [1, 2]])


## 3.2.3 比較式をブールマスクにする

列と値を比較すると、元のindexと対応したブール値のSeriesができます。`True`の行だけが残り、`mask.sum()`は該当件数になります。抽出前後の件数を表示すると、条件の誤りに気づきやすくなります。

In [ ]:
large = df["registered"] >= 30
print(large.head())
print("該当件数:", int(large.sum()))
print(df.loc[large, ["month", "centre_name", "registered"]].head())


## 3.2.4 ブール論理で複数条件を組み立てる

Pythonの単一の真偽値には`and`、`or`、`not`を使います。pandasのSeriesを行ごとに組み合わせるときは`&`、`|`、`~`を使い、各比較を括弧で囲みます。`and`へSeriesを渡すと、Series全体を一つの真偽値にできずエラーになります。

In [ ]:
truth = pd.DataFrame({"A": [False, False, True, True], "B": [False, True, False, True]})
truth["A & B"] = truth["A"] & truth["B"]
truth["A | B"] = truth["A"] | truth["B"]
truth["~A"] = ~truth["A"]
truth


### AND・OR・NOTを件数で確かめる

ANDは条件を狭め、ORは通常広げます。否定はTrueとFalseを反転します。ド・モルガンの法則により`~(A | B)`は`(~A) & (~B)`と同じですが、業務上の意味が読みやすい形を選びます。

In [ ]:
report = df.assign(
    attendance_rate=df["attended"] / df["registered"] * 100,
    completion_rate=df["completed"] / df["registered"] * 100,
)
large = report["registered"] >= 30
low_attendance = report["attendance_rate"] < 80
print("AND件数:", int((large & low_attendance).sum()))
print("OR件数:", int((large | low_attendance).sum()))


## 3.2.5 所属・範囲・欠損を条件に表す

複数候補のいずれかに一致する条件は`isin()`、下限と上限を持つ条件は`between()`で表せます。`between()`は既定で両端を含むため、境界を含めるかを問題文と一致させます。

In [ ]:
months = report["month"].isin(["2026-02", "2026-03"])
medium_size = report["registered"].between(25, 35, inclusive="both")
print(report.loc[months & medium_size, ["month", "centre_name", "registered"]])


### 欠損を条件から偶然落とさない

欠損値との大小比較は通常Falseになり、理由を示さないまま抽出から消えることがあります。値が必要な条件には`notna()`、欠損を調べる条件には`isna()`を組み込み、該当件数を別に記録します。ここでは値を補完せず、3.3で扱う品質問題として残します。

In [ ]:
has_attendance = report["attended"].notna()
low_attendance_known = has_attendance & (report["attendance_rate"] < 80)
print("出席値あり・80%未満:", int(low_attendance_known.sum()))
print("出席値欠損:", int(report["attended"].isna().sum()))


## 3.2.6 抽出し、並べ、件数を検証する

問いを名前付きマスクへ分け、最後に`loc`で組み合わせると、条件を一つずつ検証できます。抽出結果を勝手に「全データ」と呼ばず、元件数、該当件数、条件を一緒に記録します。

In [ ]:
selected_columns = ["month", "centre_id", "centre_name", "registered", "attendance_rate", "completion_rate"]
priority_mask = (
    report["month"].isin(["2026-02", "2026-03"])
    & (report["registered"] >= 30)
    & report["attended"].notna()
    & (report["attendance_rate"] < 80)
)
priority = report.loc[priority_mask, selected_columns].sort_values(["month", "centre_id"])
print("元の行数:", len(report), "抽出行数:", len(priority))
priority


### マスクと行をindexで対応させる

pandasはブールマスクを位置だけでなくindexラベルで対応させます。別のDataFrameから作ったマスクや、indexを変更した後の古いマスクを流用すると、ずれやエラーの原因になります。原則として抽出対象と同じDataFrameからマスクを作ります。

## 3.2.7 統合練習：別の問いを条件へ翻訳する

「2026年2月または3月、Python Foundations、登録者25～40人、修了率75%未満、修了値が欠損していないセンター」を抽出してください。必要列を検証し、各部分マスクと最終マスクの件数を表示し、結果を月・センターID順に並べます。条件の否定を一つ含む別の問いも作って比較してください。

In [ ]:
# ここに応用練習の解答を書きます。


## まとめ

- 問いを、表示する列と判定に使う部分条件へ分けました。
- 名前を付けたブールマスクを組み合わせ、locで行と列を選びました。
- 欠損と境界を明示し、抽出件数と並び順を検証しました。

## 次のレッスンへ

再現可能な抽出条件を作れるようになりました。3.3では、抽出前のデータに含まれる欠損、表記ゆれ、矛盾、重複を、根拠を残しながら扱います。

**学習時間の目安:** 約4時間
